## Imports

### Install packages

In [2]:
if False:
    !sudo /bin/bash -c "(source /venv/bin/activate; pip install --quiet jupyterlab-vim)"
    !jupyter labextension enable

### Import modules

In [3]:
%load_ext autoreload
%autoreload 2

import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ipywidgets import interact, FloatSlider, IntSlider, widgets
from IPython.display import display, Markdown

# Set plotting style.
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [7]:
import msml610_utils as ut
import utils_Lesson94_Information_Theory as utils

ut.config_notebook()

# Initialize logger.
logging.basicConfig(level=logging.INFO)
_LOG = logging.getLogger(__name__)

# Setting notebook style
# Notebook signature
Python 3.12.3
Linux 0e2bb8a7cac3 6.10.14-linuxkit #1 SMP Tue Apr 15 16:00:54 UTC 2025 aarch64 aarch64 aarch64 GNU/Linux
numpy version=1.26.4
pymc version=5.18.2
matplotlib version=3.10.3
arviz version=0.21.0
preliz version=0.19.0


<a name='entropy'></a>
# 1. Entropy and Uncertainty

## Definition

**Entropy** $H(X)$ of a discrete random variable $X$ is defined as:

$$H(X) = -\sum_x p(x) \log_2 p(x)$$

## Intuition

- Entropy quantifies the average level of **information**, **surprise**, or **uncertainty** inherent in the variable's possible outcomes
- High entropy = more unpredictability
- Low entropy = more certainty
- Unit: bits (when using $\log_2$)

## Examples

- **Fair coin**: Two equally likely outcomes → maximum uncertainty, $H = 1$ bit
- **Biased coin**: If heads occurs 90% of the time → less uncertainty, $H < 1$ bit

In [8]:
# Test with fair coin.
fair_coin = [0.5, 0.5]
print(f"Fair coin entropy: {utils.calculate_entropy(fair_coin):.4f} bits")

# Test with biased coin.
biased_coin = [0.9, 0.1]
print(f"Biased coin (90-10) entropy: {utils.calculate_entropy(biased_coin):.4f} bits")

# Test with certain outcome.
certain = [1.0, 0.0]
print(f"Certain outcome entropy: {utils.calculate_entropy(certain):.4f} bits")

Fair coin entropy: 1.0000 bits
Biased coin (90-10) entropy: 0.4690 bits
Certain outcome entropy: -0.0000 bits


## Interactive Visualization: Binary Entropy

Use the slider below to adjust the probability $p$ of a binary random variable and observe how entropy changes.

In [4]:
# Create interactive widget.
interact(utils.plot_binary_entropy_interactive,
         p=FloatSlider(min=0.01, max=0.99, step=0.01, value=0.5,
                      description='Probability p:', style={'description_width': 'initial'}));

NameError: name 'utils' is not defined

<a name='joint-conditional'></a>
# 2. Joint and Conditional Entropy

## Joint Entropy

**Joint entropy** $H(X, Y)$ of two variables $X$ and $Y$:

$$H(X, Y) = -\sum_{x,y} p(x,y) \log_2 p(x,y)$$

- Describes the information needed for the joint distribution of $X$ and $Y$
- For independent variables: $H(X, Y) = H(X) + H(Y)$

## Conditional Entropy

**Conditional entropy** $H(Y|X)$ measures uncertainty in $Y$ after observing $X$:

$$H(Y|X) = -\sum_{x,y} p(x,y) \log_2 p(y|x) = \sum_x p(x) H(Y|X=x)$$

**Properties:**
- Low $H(Y|X)$ implies $X$ has strong predictive power for $Y$
- If $Y = X$, then $H(Y|X) = 0$ (no uncertainty)
- If $X$ and $Y$ are independent, then $H(Y|X) = H(Y)$

## Chain Rule for Entropy

$$H(X, Y) = H(X) + H(Y|X) = H(Y) + H(X|Y)$$

In [ ]:
# Example: Weather and Activity.
# X = Weather (0: Sunny, 1: Rainy)
# Y = Activity (0: Park, 1: Cinema)
joint_prob = np.array([
    [0.35, 0.15],  # Sunny: 35% park, 15% cinema
    [0.10, 0.40]   # Rainy: 10% park, 40% cinema
])

print("Joint Probability Distribution:")
print(pd.DataFrame(joint_prob, 
                   index=['Sunny', 'Rainy'], 
                   columns=['Park', 'Cinema']))
print()

# Calculate marginals.
p_weather = joint_prob.sum(axis=1)
p_activity = joint_prob.sum(axis=0)

h_weather = utils.calculate_entropy(p_weather)
h_activity = utils.calculate_entropy(p_activity)
h_joint = utils.calculate_joint_entropy(joint_prob)
h_activity_given_weather = utils.calculate_conditional_entropy(joint_prob)

print(f"H(Weather) = {h_weather:.4f} bits")
print(f"H(Activity) = {h_activity:.4f} bits")
print(f"H(Weather, Activity) = {h_joint:.4f} bits")
print(f"H(Activity|Weather) = {h_activity_given_weather:.4f} bits")
print()
print(f"Chain rule verification: H(Weather) + H(Activity|Weather) = {h_weather + h_activity_given_weather:.4f} bits")
print(f"Should equal H(Weather, Activity) = {h_joint:.4f} bits")

<a name='mutual-info'></a>
# 3. Mutual Information

## Definition

**Mutual information** $I(X;Y)$ measures how much knowing one variable reduces uncertainty about the other:

$$I(X;Y) = H(X) - H(X|Y) = H(Y) - H(Y|X) = H(X) + H(Y) - H(X,Y)$$

## Intuition

- Measures the shared information between two variables
- Quantifies the reduction in uncertainty about one variable given the other
- Symmetric: $I(X;Y) = I(Y;X)$

## Properties

- Non-negative: $I(X;Y) \geq 0$
- $I(X;Y) = 0$ if and only if $X$ and $Y$ are independent
- $I(X;X) = H(X)$ (self-information equals entropy)
- Higher mutual information indicates stronger relationship

## Applications

- Feature selection in machine learning
- Dimensionality reduction
- Dependency detection in data

In [ ]:
# Use the weather-activity example.
print("Example: Weather and Activity Correlation")
print("=" * 50)
utils.visualize_information_decomposition(joint_prob)

# Calculate and display mutual information.
mi = utils.calculate_mutual_information(joint_prob)
print(f"\nMutual Information I(Weather; Activity) = {mi:.4f} bits")
print(f"This means knowing the weather reduces uncertainty about activity by {mi:.4f} bits")

## Interactive Visualization: Correlation and Mutual Information

Adjust the correlation strength to see how it affects mutual information between two variables.

In [ ]:
# Create interactive widget.
interact(utils.plot_mutual_info_interactive,
         correlation=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5,
                                description='Correlation:', style={'description_width': 'initial'}));

<a name='kl-cross'></a>
# 4. KL Divergence and Cross-Entropy

## KL Divergence

**Kullback-Leibler (KL) Divergence** $D_{KL}(P \| Q)$ measures how one distribution differs from another:

$$D_{KL}(P \| Q) = \sum_x P(x) \log_2 \frac{P(x)}{Q(x)}$$

**Properties:**
- Not symmetric: $D_{KL}(P \| Q) \neq D_{KL}(Q \| P)$
- Non-negative: $D_{KL}(P \| Q) \geq 0$
- $D_{KL}(P \| Q) = 0$ if and only if $P = Q$
- Not a metric (no triangle inequality)

**Intuition:** Quantifies how much information is lost when $Q$ is used to approximate $P$

## Cross-Entropy

**Cross-entropy** $H(P, Q)$ measures the average number of bits needed to encode data from $P$ using code optimized for $Q$:

$$H(P, Q) = -\sum_x P(x) \log_2 Q(x)$$

**Relationship:**
$$H(P, Q) = H(P) + D_{KL}(P \| Q)$$

**Applications:**
- Loss function in classification (logistic regression, neural networks)
- Model evaluation and comparison
- Information compression

In [ ]:
# Example: Classification problem.
# True distribution (ground truth labels).
true_dist = np.array([0.0, 0.0, 1.0, 0.0])  # Class 2 is correct.

# Model predictions (different confidence levels).
model_confident = np.array([0.05, 0.05, 0.85, 0.05])  # Confident and correct.
model_uncertain = np.array([0.25, 0.25, 0.25, 0.25])  # Uncertain (uniform).
model_wrong = np.array([0.05, 0.85, 0.05, 0.05])     # Confident but wrong.

print("Classification Example")
print("=" * 60)
print("True label: Class 2")
print()

for name, model_pred in [("Confident & Correct", model_confident),
                         ("Uncertain", model_uncertain),
                         ("Confident & Wrong", model_wrong)]:
    kl = utils.calculate_kl_divergence(true_dist, model_pred)
    ce = utils.calculate_cross_entropy(true_dist, model_pred)
    h_true = utils.calculate_entropy(true_dist)
    print(f"{name}:")
    print(f"  Model prediction: {model_pred}")
    print(f"  Cross-Entropy: {ce:.4f} bits")
    print(f"  KL Divergence: {kl:.4f} bits")
    print(f"  H(P) + D_KL(P||Q) = {h_true:.4f} + {kl:.4f} = {h_true + kl:.4f} (should equal CE)")
    print()

## Interactive Visualization: KL Divergence and Distribution Comparison

Adjust the parameters to see how KL divergence changes when approximating a distribution.

In [ ]:
# Create interactive widget.
interact(utils.plot_kl_divergence_interactive,
         p1=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.7,
                       description='P(outcome=1):', style={'description_width': 'initial'}),
         q1=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.5,
                       description='Q(outcome=1):', style={'description_width': 'initial'}));

<a name='advanced'></a>
# 5. Advanced Topics

## Data Processing Inequality

**Statement:** Processing data cannot increase information, it can only lose information.

Formally, if $X \to Y \to Z$ forms a Markov chain, then:
$$I(X;Z) \leq I(X;Y)$$

**Intuition:** Information can only be lost through processing, never gained.

**Example:** If $X$ is a raw image and $Y$ is compressed version, no further processing $Z$ will recover more information about $X$ than $Y$ already contains.

## Maximum Entropy Principle

**Principle:** Use the distribution with the largest entropy given the constraints.

**Examples of maximum entropy distributions:**
- No constraints → Uniform distribution
- Positive mean constraint → Exponential distribution  
- Fixed variance → Normal distribution

## Minimum Description Length (MDL)

**Principle:** Select the model that minimizes total description length:

$$MDL(H) = L(H) + L(D | H)$$

where:
- $L(H)$ = length of the model/hypothesis
- $L(D|H)$ = length of data encoded using the model

**Intuition:** Balances model complexity with data fit (Occam's Razor principle)

## Kolmogorov Complexity

**Definition:** The length of the shortest program that outputs a string $x$.

**Examples:**
- String "00000000..." has low complexity (simple loop)
- Random string has high complexity (no pattern, no compression)

**Properties:**
- Not computable (theoretical concept)
- Related to MDL in practice
- Measures algorithmic randomness

In [ ]:
utils.demonstrate_data_processing_inequality()

# Summary and Key Takeaways

## Information Theory Relationships

The fundamental relationships in information theory:

1. **Entropy relationships:**
   - $H(X, Y) = H(X) + H(Y|X) = H(Y) + H(X|Y)$ (Chain Rule)
   - $H(X|Y) \leq H(X)$ with equality iff $X$ and $Y$ are independent

2. **Mutual information:**
   - $I(X;Y) = H(X) - H(X|Y) = H(Y) - H(Y|X)$
   - $I(X;Y) = H(X) + H(Y) - H(X,Y)$
   - $I(X;Y) \geq 0$ with equality iff $X$ and $Y$ are independent

3. **KL divergence and cross-entropy:**
   - $H(P,Q) = H(P) + D_{KL}(P \| Q)$
   - $D_{KL}(P \| Q) \geq 0$ with equality iff $P = Q$

4. **Data processing inequality:**
   - $I(X;Z) \leq I(X;Y)$ if $X \to Y \to Z$

## Applications in Machine Learning

- **Entropy:** Feature selection, decision tree splitting criteria
- **Mutual Information:** Dependency detection, feature relevance
- **Cross-Entropy:** Loss function for classification
- **KL Divergence:** Variational inference, model comparison
- **MDL:** Model selection, preventing overfitting

# Exercises

Try these exercises to deepen your understanding:

1. **Entropy Exercise:** Calculate the entropy of a 6-sided fair die. What happens if one outcome has probability 0.5 and the others share the remaining probability equally?

2. **Mutual Information Exercise:** Given the joint distribution in the weather-activity example, calculate $I(Weather; Activity)$ manually and verify it matches the computed value.

3. **Cross-Entropy Exercise:** For a 3-class classification problem, compute the cross-entropy loss when:
   - True label: class 1
   - Model prediction: [0.2, 0.6, 0.2]
   
4. **Data Processing Exercise:** Explain why JPEG compression is lossy in terms of the data processing inequality.

5. **Maximum Entropy Exercise:** Show that the uniform distribution maximizes entropy among all distributions with the same number of outcomes.

In [ ]:
# Exercise workspace - use this cell to work on the exercises above.

# Exercise 1: Fair die entropy.
die_probs = np.array([1/6] * 6)
die_entropy = utils.calculate_entropy(die_probs)
print(f"Exercise 1: Fair die entropy = {die_entropy:.4f} bits")

# Your code here for other exercises...